# LLM Evaluation 4 of 4: Turn Confidence into Abstention Policy

**Guiding question:** When Riverside says `0.8`, does that score correspond to roughly 80%
correctness, and what coverage must it give up to meet a risk target?

This notebook owns reliability, ECE, temperature scaling, risk-coverage, and abstention
policy. Unsupported-claim detection supplies an independent risk signal from Part 3; it is
not retaught here.


| Part | Complaint | Measured evidence |
| --- | --- | --- |
| 1 | Accuracy does not make confidence meaningful | Reliability bins and ECE |
| 2 | Scores are overconfident | Held-out temperature scaling |
| 3 | Better precision costs coverage | Risk-coverage curve |
| 4 | Confidence alone can serve fluent fabrications | Two-gate routing record |

![Static reliability and calibration intuition](images/04-calibration-and-confidence-theory-01.png)


In [ ]:
# Deterministic labeled confidence fixture.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from evaluation_shared import COURSE_CAVEAT, set_seed

set_seed()
confidence = np.array([.97,.94,.91,.88,.84,.81,.78,.74,.70,.66,.61,.58,.54,.49,.44,.39,.34,.29,.24,.18])
correct = np.array([1,1,0,1,0,1,1,0,1,0,1,0,1,0,0,1,0,0,0,0], dtype=int)
split = 10
print(COURSE_CAVEAT)
print("accuracy=", correct.mean(), "mean_confidence=", confidence.mean())


## Part 1 - Reliability before one-number summaries

**Predict:** Which confidence region is most overconfident? ECE will summarize the weighted
gaps, but the reliability table must retain their direction.


In [ ]:
# Reliability bins and expected calibration error.
def reliability_table(scores, labels, bins=5):
    edges = np.linspace(0, 1, bins + 1)
    rows = []
    for index in range(bins):
        lower, upper = edges[index], edges[index + 1]
        mask = (scores >= lower) & (scores < upper if index < bins - 1 else scores <= upper)
        if not mask.any():
            continue
        rows.append({
            "lower": lower, "upper": upper, "count": int(mask.sum()),
            "mean_confidence": float(scores[mask].mean()),
            "accuracy": float(labels[mask].mean()),
        })
    table = pd.DataFrame(rows)
    table["gap"] = table["mean_confidence"] - table["accuracy"]
    table["weight"] = table["count"] / len(scores)
    return table

reliability = reliability_table(confidence, correct)
ece = float((reliability["weight"] * reliability["gap"].abs()).sum())
display(reliability.round(3))
print(f"ECE={ece:.3f}")


In [ ]:
# Static reliability diagram keeps local failures visible.
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="black", label="perfect calibration")
ax.scatter(reliability["mean_confidence"], reliability["accuracy"],
           s=60 + reliability["count"] * 15, label="observed bins")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="mean confidence", ylabel="observed accuracy")
ax.set_title("Reliability before calibration")
ax.legend()
plt.tight_layout()
plt.show()


## Part 2 - Temperature scaling changes magnitude, not ranking

Fit one positive temperature on the calibration half, freeze it, then evaluate the test half.
The selected answer never changes because all logits are rescaled uniformly.


In [ ]:
# Convert binary confidence into logits and fit temperature by negative log-likelihood.
eps = 1e-6
logits = np.log(np.clip(confidence, eps, 1 - eps) / np.clip(1 - confidence, eps, 1 - eps))

def sigmoid(values):
    return 1 / (1 + np.exp(-values))

def nll(temperature):
    probabilities = sigmoid(logits[:split] / temperature)
    labels = correct[:split]
    return -np.mean(
        labels * np.log(np.clip(probabilities, eps, 1))
        + (1 - labels) * np.log(np.clip(1 - probabilities, eps, 1))
    )

fitted = minimize_scalar(nll, bounds=(0.2, 8.0), method="bounded").x
calibrated = sigmoid(logits / fitted)
before_test = reliability_table(confidence[split:], correct[split:], bins=4)
after_test = reliability_table(calibrated[split:], correct[split:], bins=4)
ece_before = float((before_test["weight"] * before_test["gap"].abs()).sum())
ece_after = float((after_test["weight"] * after_test["gap"].abs()).sum())
print(f"temperature={fitted:.3f}; test ECE {ece_before:.3f} -> {ece_after:.3f}")


In [ ]:
# Verify rank preservation and compare magnitudes.
comparison = pd.DataFrame({
    "raw": confidence[split:],
    "calibrated": calibrated[split:],
    "correct": correct[split:],
})
display(comparison.round(3))
assert np.array_equal(np.argsort(-confidence), np.argsort(-calibrated))
print("Ranking preserved; temperature cannot repair poor discrimination.")


## Part 3 - Abstention buys lower risk with lower coverage

A threshold without coverage is incomplete policy evidence.

![Threshold movement across selective prediction](images/selective-prediction-threshold-sweep.gif)


In [ ]:
# Sweep thresholds on the held-out test split.
test_scores, test_labels = calibrated[split:], correct[split:]
risk_rows = []
for threshold in np.linspace(0.1, 0.9, 17):
    served = test_scores >= threshold
    coverage = served.mean()
    risk = 0.0 if not served.any() else 1 - test_labels[served].mean()
    risk_rows.append({
        "threshold": threshold, "coverage": coverage,
        "selective_risk": risk, "served": int(served.sum()),
    })
risk_coverage = pd.DataFrame(risk_rows)
display(risk_coverage.round(3))


In [ ]:
# Static risk-coverage evidence remains available without animation support.
ordered = risk_coverage.sort_values("coverage")
ax = ordered.plot(x="coverage", y="selective_risk", marker="o", figsize=(7, 4))
ax.set(xlim=(0, 1.05), ylim=(0, 1.05))
ax.set_title("Risk-coverage frontier on held-out cases")
plt.tight_layout()
plt.show()

eligible = risk_coverage.query("selective_risk <= 0.20")
chosen = eligible.sort_values("coverage", ascending=False).iloc[0]
print("Highest coverage meeting risk <= 0.20:", chosen.to_dict())


## Part 4 - Confidence and claim risk are independent gates

A fluent fabrication can be high-confidence. Part 3 supplies claim risk; this notebook turns
calibrated confidence plus that risk into a routing policy.


In [ ]:
# Two-gate routing records preserve both signals and the reason.
routing_cases = pd.DataFrame([
    {"id": "grounded-high", "confidence": 0.82, "claim_risk": 0.10},
    {"id": "fabricated-high", "confidence": 0.91, "claim_risk": 0.92},
    {"id": "grounded-low", "confidence": 0.38, "claim_risk": 0.12},
    {"id": "ambiguous", "confidence": 0.57, "claim_risk": 0.48},
])

def route(row):
    if row["claim_risk"] >= 0.70:
        return "HOLD_FOR_REVIEW"
    if row["confidence"] >= chosen["threshold"] and row["claim_risk"] < 0.30:
        return "SERVE"
    if row["confidence"] >= 0.40:
        return "SERVE_WITH_CAVEAT"
    return "REFUSE"

routing_cases["route"] = routing_cases.apply(route, axis=1)
display(routing_cases)
assert routing_cases.query("id == 'fabricated-high'").iloc[0]["route"] == "HOLD_FOR_REVIEW"


## Scorecard

- Reliability bins exposed where confidence and observed accuracy diverged.
- ECE summarized weighted gaps without replacing the diagram.
- Temperature scaling changed probability magnitude and preserved ranking.
- Risk-coverage made the abstention cost visible.
- Claim risk overrode high confidence for a fluent fabrication.

Choose thresholds from validated risk and coverage costs, then monitor shift by slice. The
tiny fixture demonstrates mechanics; it does not establish a deployable confidence model.
